# Train YOLO-seg on Kaggle — 4-fold CV + fixed-test evaluation

Runs the Ultralytics YOLO instance-segmentation baselines (**yolo11** = proven,
**yolo26** = latest) under the *same* protocol as TransUNet: 4-fold CV, fixed
held-out test set, and per-class / per-position / per-tooth-type **IoU + Dice**.

YOLO predicts one mask per tooth; `evaluate_yolo_seg.py` rasterizes those back
into a 33-class label map and reuses the exact reporting code from the dense
segmenters, so each fold's `test_summary.json` is directly comparable to
TransUNet's.

Like TransUNet, YOLO-seg is **image-only** — it needs only `data/splits` (img +
`masks_semantic`), not `data/bb_maps/`. The same `splits.zip` you uploaded for
TransUNet works here.

## Before you run
1. **Accelerator:** Settings → Accelerator → **GPU T4 x2** or **GPU P100**.
2. **Internet:** Settings → Internet → **On** (needed for `git clone` + `pip` + weight download).
3. **Data:** add your `pbl4-splits` dataset (right panel → *Add Input*) — the
   same one used for TransUNet. It must contain `splits/folds/fold_0..3/`,
   `splits/test/` and `splits/class_map.txt`.

Run the cells top to bottom. Two models × 4 folds is the heavy part; if a session
is cut short, just re-run — completed folds are skipped/overwritten cleanly.

## 1. Configure

In [ ]:
SPLITS_PATH = "/kaggle/input/pbl4-splits/splits"      # auto-detected in Step 3 if wrong
REPO_URL    = "https://github.com/Huay0804/PBL4.git"  # private? https://<TOKEN>@github.com/...
REPO_DIR    = "/kaggle/working/PBL4"

MODELS = ["yolo11", "yolo26"]  # drop to ["yolo11"] for just the proven baseline
SIZE   = "m"                    # n/s/m/l/x — m fits T4/P100 comfortably

## 2. Get the code and install Ultralytics
Unlike the TransUNet notebook (which pins to Kaggle's fragile TF/numpy stack),
Ultralytics installs cleanly on top of the preinstalled PyTorch.

In [ ]:
import os, subprocess, sys

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
print("Working dir:", os.getcwd())

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics"], check=True)

import torch, ultralytics
print(f"ultralytics {ultralytics.__version__} | torch {torch.__version__} | CUDA {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU — confirm the accelerator is on (right panel → Settings).")

## 3. Wire up data and materialize the YOLO-seg dataset views
Links the read-only dataset to `data/splits`, then converts every fold's masks
into YOLO polygon labels under `data/yolo_seg/` (pure file I/O, ~1 min).

In [ ]:
import os, glob, shutil
os.chdir(REPO_DIR)
sys.path.insert(0, os.path.abspath("scripts"))

def _resolve_splits():
    if os.path.exists(os.path.join(SPLITS_PATH, "class_map.txt")):
        return SPLITS_PATH
    hits = glob.glob("/kaggle/input/**/class_map.txt", recursive=True)
    if not hits:
        raise FileNotFoundError("No class_map.txt under /kaggle/input — did you Add the dataset?")
    if len(hits) > 1:
        print(f"Multiple candidates: {hits}\nUsing first — set SPLITS_PATH to override.")
    return os.path.dirname(hits[0])

splits_path = _resolve_splits()
os.makedirs("data", exist_ok=True)
link = "data/splits"
if os.path.islink(link): os.remove(link)
elif os.path.isdir(link): shutil.rmtree(link)
elif os.path.exists(link): os.remove(link)
os.symlink(splits_path, link)
print(f"Linked data/splits -> {splits_path}")

rc = subprocess.run([sys.executable, "scripts/prepare_yolo_seg_data.py"]).returncode
if rc != 0:
    raise SystemExit("Dataset materialization failed.")

## 4. Train all models × 4 folds
yolo11 downloads COCO-pretrained `-seg` weights automatically. yolo26 trains
from its `-seg` config. Checkpoints land at
`runs/cv/fold_<k>/<model>_seg/weights/best.pt`.

In [ ]:
import os, subprocess
os.chdir(REPO_DIR)

for model in MODELS:
    for k in range(4):
        print(f"\n========== TRAIN {model} fold {k} ==========", flush=True)
        rc = subprocess.run(
            f"python -u scripts/train_yolo_seg_cv.py --model {model} --size {SIZE} "
            f"--fold {k} --skip-prepare",
            shell=True,
        ).returncode
        if rc != 0:
            raise SystemExit(f"{model} fold {k} training failed (exit {rc}).")
print("\nAll models/folds trained.")

## 5. Evaluate each fold on the fixed test set
Writes `test_summary.json`, `test_metrics.json`, `per_class_*`, `per_position_*`,
`per_tooth_type_*` and `per_quadrant_*` next to each checkpoint — same format as
the dense segmenters.

In [ ]:
import os, subprocess
os.chdir(REPO_DIR)

for model in MODELS:
    for k in range(4):
        print(f"\n========== EVAL {model} fold {k} ==========", flush=True)
        rc = subprocess.run(
            f"python -u scripts/evaluate_yolo_seg.py --model {model} --size {SIZE} --cv-fold {k}",
            shell=True,
        ).returncode
        if rc != 0:
            raise SystemExit(f"{model} fold {k} evaluation failed (exit {rc}).")
print("\nAll folds evaluated.")

## 6. Results — cross-validation aggregate per model
Mean ± std of the fixed-test macro/weighted IoU & Dice across the 4 folds, ready
to sit next to TransUNet's numbers.

In [ ]:
import glob, json, os
import numpy as np
os.chdir(REPO_DIR)

KEYS = ["macro_iou", "macro_dice", "weighted_iou", "weighted_dice"]
for model in MODELS:
    run_name = f"{model}_seg"
    summaries = sorted(glob.glob(f"runs/cv/fold_*/{run_name}/test_summary.json"))
    if not summaries:
        print(f"[{model}] no test summaries found"); continue
    rows = [json.load(open(p)) for p in summaries]
    print(f"\n=== {run_name}: {len(rows)} folds ===")
    agg = {}
    for key in KEYS:
        vals = np.array([r[key] for r in rows], float)
        agg[key] = {"mean": float(vals.mean()), "std": float(vals.std())}
        print(f"  {key:14s}: {vals.mean():.4f} ± {vals.std():.4f}")
    out = f"runs/cv/{run_name}_cv_summary.json"
    json.dump({"model": model, "folds": len(rows), "aggregate": agg,
               "per_fold": rows}, open(out, "w"), indent=2)
    print(f"  wrote {out}")

## 7. Download results
Zips the CV outputs (checkpoints + metrics) to the **Output** tab, excluding
bulky training plots and the per-epoch weights.

In [ ]:
import os
os.chdir(REPO_DIR)
!zip -r -q /kaggle/working/yolo_seg_runs.zip runs/cv -x "*/weights/last.pt" "*/weights/epoch*.pt"
print("Wrote /kaggle/working/yolo_seg_runs.zip — download it from the Output tab.")